# Convex Cost Experiment

Polars-based consolidation of each experiment under `logs/`, comparing the Longstaff-Schwartz Monte Carlo (LSM) benchmark price with the best-performing RL evaluation episode.

> Compatibility: validated inside the `EP11` conda environment (Python 3.11, NumPy 2.2.6, pandas 2.3.0) with `polars 1.35.1`.

In [1]:
from pathlib import Path
import polars as pl
import pandas as pd
from tqdm.notebook import tqdm

pd.options.display.float_format = "{:,.6f}".format

RISK_FREE_RATE = 0.05
MATURITY = 0.0833
N_RIGHTS = 22
DELTA_T = MATURITY / (N_RIGHTS - 1)


def locate_logs_dir(start_dir: Path | None = None) -> Path:
    start_dir = Path(start_dir or Path.cwd()).resolve()
    for current in (start_dir, *start_dir.parents):
        candidate = current / "logs"
        if candidate.exists():
            return candidate
    raise FileNotFoundError("Unable to find a 'logs' directory relative to this notebook.")


LOGS_DIR = locate_logs_dir()
LOGS_DIR

PosixPath('/Users/alexanderithakis/Documents/GitHub/DRL-Swing-Options/logs')

In [2]:
discount_expr = (-RISK_FREE_RATE * DELTA_T * pl.col("time_step")).exp()


def compute_lsm_price(lsm_csv: Path) -> float:
    df = pl.read_csv(lsm_csv)
    discounted = df.with_columns(
        (pl.col("payoff") * discount_expr).alias("discounted_payoff")
    )
    path_totals = (
        discounted.group_by("path")
        .agg(pl.col("discounted_payoff").sum())
        .select(pl.col("discounted_payoff"))
    )
    return float(path_totals.mean().item())


def _extract_episode_number(stem: str) -> int | None:
    try:
        return int(stem.split("_")[-1])
    except ValueError:
        return None


def compute_best_rl_price(rl_csv_files: list[Path]) -> tuple[float | None, int | None]:
    best_price = None
    best_episode = None
    for rl_csv in rl_csv_files:
        df = pl.read_csv(rl_csv)
        episode_price = (
            df.group_by("path")
            .agg(pl.col("reward").sum().alias("reward_sum"))
            .select(pl.col("reward_sum").mean())
            .item()
        )
        if episode_price is None:
            continue
        if best_price is None or episode_price > best_price:
            best_price = float(episode_price)
            best_episode = _extract_episode_number(rl_csv.stem)
    return best_price, best_episode

In [3]:
experiment_rows: list[dict[str, object]] = []
experiment_dirs = sorted(LOGS_DIR.iterdir())
for experiment_dir in tqdm(experiment_dirs, total=len(experiment_dirs)):
    eval_dir = experiment_dir / "evaluations"
    lsm_file = eval_dir / "lsm.csv"
    rl_files = sorted(eval_dir.glob("rl_episode_*.csv"))
    if not eval_dir.is_dir() or not lsm_file.exists() or not rl_files:
        continue

    lsm_price = compute_lsm_price(lsm_file)
    best_rl_price, best_episode_number = compute_best_rl_price(rl_files)

    experiment_rows.append(
        {
            "name": experiment_dir.name,
            "LSM": lsm_price,
            "Best RL": best_rl_price,
            "Best RL Episode": best_episode_number,
        }
    )

summary_df = pl.DataFrame(experiment_rows).sort("name")
summary_view = summary_df.select(["name", "LSM", "Best RL", "Best RL Episode"])
summary_view.to_pandas()

  0%|          | 0/44 [00:00<?, ?it/s]

,name,LSM,Best RL,Best RL Episode
0,SwingOption_20_11,2.625979,2.588738,10240
1,SwingOption_20_12,2.684629,2.538439,3072
2,SwingOption_20_13,2.651677,2.608920,12288
3,SwingOption_20_14,2.684180,2.660009,11264
4,SwingOption_20_15,2.682344,2.611791,15360
5,SwingOption_20_16,2.680587,2.643240,14336
6,SwingOption_20_17,2.664366,2.164766,1024
7,SwingOption_20_18,2.676665,2.620475,16384
8,SwingOption_20_c0.05_gamma1.5_11,1.895797,1.898412,15360
9,SwingOption_20_c0.05_gamma1.5_12,1.944535,1.962262,12288


In [4]:
sumdf = summary_view.to_pandas()

sumdf['PctDiff'] = (sumdf.loc[:,'Best RL'] - sumdf.loc[:,'LSM']) / sumdf.loc[:,'LSM'].abs() * 100
sumdf = sumdf[['name', 'LSM', 'Best RL', 'PctDiff', 'Best RL Episode']]
sumdf.set_index('name', inplace=True)
sumdf.to_csv('Convex Costs Results.csv')
sumdf

,LSM,Best RL,PctDiff,Best RL Episode
name,,,,
SwingOption_20_11,2.625979,2.588738,-1.418151,10240
SwingOption_20_12,2.684629,2.538439,-5.445450,3072
SwingOption_20_13,2.651677,2.608920,-1.612475,12288
SwingOption_20_14,2.684180,2.660009,-0.900494,11264
SwingOption_20_15,2.682344,2.611791,-2.630270,15360
SwingOption_20_16,2.680587,2.643240,-1.393257,14336
SwingOption_20_17,2.664366,2.164766,-18.751184,1024
SwingOption_20_18,2.676665,2.620475,-2.099247,16384
SwingOption_20_c0.05_gamma1.5_11,1.895797,1.898412,0.137920,15360
